# 06 - Validation and Biomarker Comparison

Explore validation results from `data/results/validation/`.  
Biomarker correlation heatmap, ROC curves with confidence intervals,
NRI/IDI tables. Critical for peer review — reviewers will want these comparisons.
**This notebook reads pre-computed results only.**

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

results_dir = os.path.join('..', config['paths']['results'], 'validation')
print('Validation results directory:', results_dir)
print('Files available:', os.listdir(results_dir) if os.path.exists(results_dir) else 'DIRECTORY NOT FOUND')

## Figure 7 - Biomarker Correlation Heatmap

Spearman correlation matrix between CSD score, DNB score, and established biomarkers
(p-tau217, Abeta42/40, NfL, GFAP). Diverging colormap centered at zero.  
Expected: moderate positive correlations (r ~ 0.2-0.5) indicating CSD adds orthogonal information.

In [ ]:
# Load correlation matrix
corr_df = pd.read_csv(os.path.join(results_dir, 'biomarker_correlations.csv'), index_col=0)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, mask=mask, square=True, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Spearman rho'})
ax.set_title('Biomarker Correlation Matrix (Spearman)')
plt.tight_layout()
plt.show()

# Check collinearity with p-tau217
if 'composite_csd_score' in corr_df.index and 'ptau217' in corr_df.columns:
    r = corr_df.loc['composite_csd_score', 'ptau217']
    print(f'CSD-ptau217 correlation: {r:.3f} ({"CAUTION: high collinearity" if abs(r) > 0.7 else "OK: orthogonal information"})')

## ROC Curves

ROC curves for each predictor at multiple time horizons (12, 24, 36 months before conversion).  
Comparison of CSD score, DNB score, p-tau217, NfL, and Abeta42/40 ratio.

In [ ]:
# Load ROC results
roc_df = pd.read_csv(os.path.join(results_dir, 'roc_results.csv'))

time_horizons = sorted(roc_df['time_horizon_months'].unique())
predictors = roc_df['predictor'].unique()

fig, axes = plt.subplots(1, len(time_horizons), figsize=(5 * len(time_horizons), 5))
if len(time_horizons) == 1:
    axes = [axes]

for i, horizon in enumerate(time_horizons):
    ax = axes[i]
    subset = roc_df[roc_df['time_horizon_months'] == horizon]

    for _, row in subset.iterrows():
        label = f"{row['predictor']} (AUC={row['auc']:.2f} [{row['auc_ci_lower']:.2f}-{row['auc_ci_upper']:.2f}])"
        # Plot diagonal reference
        ax.plot([0, 1], [0, 1], 'k--', linewidth=0.5)
        # AUC annotation as horizontal bars
        ax.barh(row['predictor'], row['auc'], height=0.6, alpha=0.7,
                xerr=[[row['auc'] - row['auc_ci_lower']], [row['auc_ci_upper'] - row['auc']]], capsize=3)

    ax.set_xlim(0.4, 1.0)
    ax.set_xlabel('AUC')
    ax.set_title(f'{horizon}-Month Prediction')
    ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.8)

plt.suptitle('Predictive Performance by Time Horizon', fontsize=13)
plt.tight_layout()
plt.show()

## Incremental Prediction: NRI and IDI

Does adding the CSD score to a model with established biomarkers improve prediction?  
Net Reclassification Improvement (NRI) and Integrated Discrimination Improvement (IDI)
with bootstrap 95% confidence intervals.

In [ ]:
# Load incremental prediction results
incremental = pd.read_csv(os.path.join(results_dir, 'incremental_prediction.csv'))

print('=== Incremental Prediction Analysis ===')
print(f'Base model: age + sex + APOE4 + established biomarkers')
print(f'Augmented model: base + composite CSD score')
print()

for _, row in incremental.iterrows():
    metric = row['metric']
    est = row['estimate']
    ci_lo = row['ci_lower']
    ci_hi = row['ci_upper']
    p = row['p_value']
    sig = '*' if p < 0.05 else ''
    print(f'  {metric}: {est:.3f} (95% CI: {ci_lo:.3f} to {ci_hi:.3f}), p = {p:.2e} {sig}')

print()
print('* = significant at alpha=0.05')

## Biomarker-Negative Subgroup Analysis

Among participants who are amyloid-negative or below p-tau217 threshold,
does CSD score still predict conversion? This is the most clinically important analysis.

In [ ]:
# Load subgroup analysis results
subgroup_path = os.path.join(results_dir, 'biomarker_negative_subgroup.csv')
if os.path.exists(subgroup_path):
    subgroup = pd.read_csv(subgroup_path)
    print('=== Biomarker-Negative Subgroup Analysis ===')
    print(subgroup.to_string(index=False))
else:
    print('Subgroup analysis results not found. Run the validation pipeline first.')